<a href="https://colab.research.google.com/github/meryambutt123-a11y/urdu-ocr-codesaviours-si26-maryam/blob/main/SI26-Week4-Maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install transformers tiktoken protobuf --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 32.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>

In [5]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor, AutoImageProcessor, AutoTokenizer
import torch

# 1. Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# 2. Safely load the Image Processor (no deprecated commands)
print("Loading explicit image processor...")
image_processor = AutoImageProcessor.from_pretrained('microsoft/trocr-base-printed')

# 3. Safely load the Tokenizer directly from RoBERTa to bypass the v5 bug
print("Loading explicit tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('roberta-large')

# 4. Stitch them into the TrOCRProcessor
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

# 5. Load the Model Weights
print("Loading model weights...")
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

# 6. Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda
Loading explicit image processor...
Loading explicit tokenizer...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


In [17]:
# ---  DATASET (RAW IMAGES FIX) ---
import pandas as pd
import torch
from torch.utils.data import Dataset, random_split
from PIL import Image
import os

class UrduOCRDataset(Dataset):
    def __init__(self, csv_file, root_dir, processor, max_target_length=128):
        self.df = pd.read_csv(csv_file).dropna()
        self.root_dir = root_dir
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Grab the exact relative path from the CSV (e.g., 'data/raw/newspaper/newspaper_28.png')
        relative_path = self.df.iloc[idx, 0]

        # Combine it directly with the project root, bypassing the 'processed' folder entirely
        img_path = os.path.join(self.root_dir, relative_path)

        image = Image.open(img_path).convert("RGB")
        text = self.df.iloc[idx, 1]

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True
        ).input_ids

        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

# --- THE FIX: Point ONLY to the main project folder ---
CSV_PATH = '/content/drive/MyDrive/Urdu_OCR_Project/data/labels.csv'
PROJECT_ROOT = '/content/drive/MyDrive/Urdu_OCR_Project/'

# Instantiate the dataset
full_dataset = UrduOCRDataset(csv_file=CSV_PATH, root_dir=PROJECT_ROOT, processor=processor)

# Dynamic 80/20 split
total_samples = len(full_dataset)
train_size = int(0.8 * total_samples)
test_size = total_samples - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

print("Dataset successfully loaded pointing to RAW images!")

Dataset successfully loaded pointing to RAW images!


In [18]:
# ---  DATALOADER & OPTIMIZER ---
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 40
Ready to train!


In [19]:
# --- TRAINING LOOP ---
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-'*30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        # Backward pass (updating weights)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            print(f'Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/3
------------------------------
Batch 0/40 | Loss: 17.2922
Batch 10/40 | Loss: 5.6155
Batch 20/40 | Loss: 3.9802
Batch 30/40 | Loss: 3.6128
Epoch 1 complete | Average Loss: 5.2312

Epoch 2/3
------------------------------
Batch 0/40 | Loss: 3.7469
Batch 10/40 | Loss: 3.6075
Batch 20/40 | Loss: 3.4964
Batch 30/40 | Loss: 3.5884
Epoch 2 complete | Average Loss: 3.6122

Epoch 3/3
------------------------------
Batch 0/40 | Loss: 3.5696
Batch 10/40 | Loss: 3.5923
Batch 20/40 | Loss: 3.4881
Batch 30/40 | Loss: 3.5341
Epoch 3 complete | Average Loss: 3.5272

Training complete!


In [20]:
# ---  SAVE THE MODEL ---
import os

# Define the save path in your Drive
save_directory = '/content/drive/MyDrive/Urdu_OCR_Project/models/trocr_urdu_raw_baseline'

# Create the folder if it doesn't exist
os.makedirs(save_directory, exist_ok=True)

# Save the fine-tuned model and processor
model.save_pretrained(save_directory)
processor.save_pretrained(save_directory)

print(f"Success! Model weights and processor saved securely to:\n{save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Model weights and processor saved securely to:
/content/drive/MyDrive/Urdu_OCR_Project/models/trocr_urdu_raw_baseline


In [22]:
# --- CELL 5: IMPROVED TEST PREDICTIONS ---
model.eval()

# Grab a single batch
test_batch = next(iter(test_loader))
pixel_values = test_batch['pixel_values'].to(device)

# Generate with explicit settings to avoid blank/padding outputs
with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_length=128,
        num_beams=4,              # Beam search helps find better character sequences
        early_stopping=True,
        no_repeat_ngram_size=2    # Prevents repeating characters
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

print("-" * 30)
for i, text in enumerate(generated_text):
    print(f"Image {i + 1} Prediction: {text}")
print("-" * 30)

------------------------------
Image 1 Prediction: �ی��� ��� ،�ڌ � ��و۩ہے�ا�ر�م�
Image 2 Prediction: ���ی�و���ا� ہ�ر۩� ��م��ے�ن�ل� م� ��
Image 3 Prediction: ����ڌی����ک۩�،���ا� ��و�ر� �م��ځ�ل�ن� مڒ�ا��
Image 4 Prediction: �ی��� � ��،ڌ� � ،�۩ے�و�اہ�ر۾ۺ�ن�م۬�
------------------------------
